# ⚙️ Terminal 1 — Streams Processor
Consumes from `raw-data`, runs ML prediction, publishes to `predictions`.

**Run this FIRST** before the producer or consumer.

In [ ]:
# confluent-kafka installs instantly on Python 3.12 — no build errors
!pip install confluent-kafka scikit-learn joblib --quiet
print('Install done!')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ════════════════════════════════════════════════
#   FILL IN YOUR CONFLUENT CLOUD CREDENTIALS HERE
# ════════════════════════════════════════════════
BOOTSTRAP_SERVERS = 'pkc-XXXXX.us-east-1.aws.confluent.cloud:9092'  # <-- replace
SASL_USERNAME     = 'YOUR_API_KEY'                                    # <-- replace
SASL_PASSWORD     = 'YOUR_API_SECRET'                                 # <-- replace
MODEL_PATH        = '/content/drive/MyDrive/kafka-assignment/bike_model.joblib'
# ════════════════════════════════════════════════

In [ ]:
import json
import time
import joblib
import numpy as np
from confluent_kafka import Consumer, Producer, KafkaException

# ── Load model ────────────────────────────────────────────────────
artifact = joblib.load(MODEL_PATH)
model    = artifact['model']
FEATURES = artifact['features']
print('[PROCESSOR] Model loaded! Features: ' + str(FEATURES))

# ── Kafka config ──────────────────────────────────────────────────
kafka_conf = {
    'bootstrap.servers': BOOTSTRAP_SERVERS,
    'security.protocol': 'SASL_SSL',
    'sasl.mechanisms':   'PLAIN',
    'sasl.username':     SASL_USERNAME,
    'sasl.password':     SASL_PASSWORD,
}

# ── Consumer (reads raw-data) ─────────────────────────────────────
consumer_conf = dict(kafka_conf)
consumer_conf['group.id']          = 'streams-processor-group'
consumer_conf['auto.offset.reset'] = 'earliest'

consumer = Consumer(consumer_conf)
consumer.subscribe(['raw-data'])

# ── Producer (writes predictions) ────────────────────────────────
producer = Producer(kafka_conf)

def delivery_report(err, msg):
    if err:
        print('[DELIVERY ERROR] ' + str(err))

print('[PROCESSOR] Connected to Kafka. Waiting for messages on raw-data...')
print('=' * 60)

# ── Main streaming loop ───────────────────────────────────────────
try:
    while True:
        msg = consumer.poll(timeout=2.0)

        if msg is None:
            print('[WAITING] No messages yet...', end='\r')
            continue

        if msg.error():
            raise KafkaException(msg.error())

        # Parse incoming record
        record = json.loads(msg.value().decode('utf-8'))

        # Build feature vector
        feature_values = [record.get(f, 0) for f in FEATURES]
        X = np.array(feature_values).reshape(1, -1)

        # Run ML prediction
        prediction = float(model.predict(X)[0])
        actual     = record.get('cnt', None)

        # Build output record
        output = {
            'row_index'       : record.get('row_index'),
            'timestamp'       : record.get('timestamp'),
            'processed_at'    : time.time(),
            'predicted_count' : round(prediction, 1),
            'actual_count'    : actual,
            'season'          : record.get('season'),
            'hr'              : record.get('hr'),
            'temp'            : record.get('temp'),
            'weathersit'      : record.get('weathersit'),
        }

        # Send to predictions topic
        producer.produce(
            topic='predictions',
            value=json.dumps(output).encode('utf-8'),
            callback=delivery_report
        )
        producer.poll(0)

        row_idx = record.get('row_index')
        pred_r  = round(prediction, 1)
        print('[PROCESSED] row=' + str(row_idx) + ' | pred=' + str(pred_r) + ' | actual=' + str(actual))

except KeyboardInterrupt:
    print('\n[STOPPED] Processor shut down.')
finally:
    consumer.close()
    producer.flush()